# 04 Subhalo smoke test

A few minutes on tiny models to check the new training code before the real subhalo runs:

1. A model conditioned on tracer spacing (FiLM) starts out giving exactly the same output as the plain model it was copied from.
2. A tiny run with two subhalo spacings, FiLM, a curriculum and `init_from` trains, resumes and scores.
3. The spacing sweep runs.

Scores will be poor; that is expected.

Runtime: **L4 GPU** with **High-RAM**. Needs the `GITHUB_TOKEN` secret.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get('GITHUB_TOKEN')
!rm -rf /content/deepvoid
!git clone -q "https://{token}@github.com/skumagai3/deepvoid.git" /content/deepvoid
!git -C /content/deepvoid remote set-url origin https://github.com/skumagai3/deepvoid.git
!pip -q install --force-reinstall --no-deps /content/deepvoid
del token
!git -C /content/deepvoid log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. FiLM starts as the identity

In [ ]:
import numpy as np
from deepvoid.models import copy_weights, unet3d

plain = unet3d(depth=3, base_filters=8)
film = unet3d(depth=3, base_filters=8, conditioning={'units': 16, 'noise': 0.1})
copied, skipped = copy_weights(plain, film)
print(len(copied), 'layers copied;', len(skipped), 'left as built')

x = np.random.default_rng(0).normal(size=(2, 32, 32, 32, 1)).astype('float32')
s = np.full((2, 1, 1, 1, 1), 7.0, 'float32')
diff = np.abs(plain.predict_on_batch(x) - film.predict_on_batch({'density': x, 'spacing': s})).max()
print('max difference:', diff)
assert diff < 1e-5

## 1b. Batches mix spacings

Two small fake volumes filled with 3s and 10s, standing in for the two spacings. Every crop should come from its own randomly chosen volume, and its values should match the spacing it is tagged with.

In [ ]:
from deepvoid.pipeline import CropSampler
from deepvoid.split import slab_split

fields = [np.full((64, 64, 64), 3.0, 'float32'), np.full((64, 64, 64), 10.0, 'float32')]
labels = np.zeros((64, 64, 64), 'uint8')
sampler = CropSampler(fields, [labels, labels], slab_split(64, buffer=2)['train'], crop=16,
                      batch_size=16, steps_per_epoch=2, rotate=True, spacings=[3.0, 10.0])
batch, y = sampler[0]
s = batch['spacing'].numpy().ravel()
values = batch['density'].numpy()[..., 0]
print('spacing of each crop:', s)
assert len(set(s)) > 1, 'batch has a single spacing'
assert all((values[i] == s[i]).all() for i in range(len(s))), 'crop does not match its spacing'
print('ok: batches mix spacings and each crop matches its tag')

## 2. Tiny run: two spacings, FiLM, curriculum, started from a saved plain model

In [ ]:
import shutil
from pathlib import Path
from deepvoid.train import run

shutil.rmtree('/content/runs', ignore_errors=True)
Path('/content/runs').mkdir()
plain.save('/content/runs/plain.keras')

config = {
    'name': 'smoke_film',
    'output': '/content/runs',
    'data': {'inputs': ['tng300-3/subhalos/L3', 'tng300-3/subhalos/L10'],
             'labels': 'tng300-3/mask'},
    'model': {'depth': 3, 'base_filters': 8, 'conditioning': {'units': 16, 'noise': 0.1}},
    'training': {'crop': 64, 'batch_size': 4, 'steps_per_epoch': 20, 'val_batches': 2,
                 'epochs': 2, 'init_from': 'plain.keras',
                 'curriculum': {'start': [3, 1], 'end': [1, 1], 'epochs': 2},
                 'loss': {'name': 'scce', 'class_weights': 'auto'}},
    'evaluate': {'inputs': ['tng300-3/subhalos/L3', 'tng300-3/subhalos/L5',
                            'tng300-3/subhalos/L10']},
}
out = run(config)

The log above should show `initialized ... layers`, and `test_scores.json` should have three entries, including L5, which the model never trained on.

## 3. Resume for one more epoch

This should print `resuming` and start at epoch 2. The log should have `val_mcc_L3` and `val_mcc_L10` columns.

In [ ]:
config['training']['epochs'] = 3
out = run(config)
print(open(out / 'log.csv').read())

## 4. Spacing sweep

In [ ]:
from deepvoid.train import spacing_sweep
sweep = spacing_sweep(out, 'tng300-3/subhalos/L10', [3, 10])

## 5. Two input channels (raw + smoothed)

The sampler keeps channels together through rotations, and a one-epoch tiny run with a smoothed second channel trains and scores.

In [ ]:
two = [np.stack([np.full((64, 64, 64), v, 'float32'), np.full((64, 64, 64), -v, 'float32')], -1)
       for v in (3.0, 10.0)]
sampler = CropSampler(two, [labels, labels], slab_split(64, buffer=2)['train'], crop=16,
                      batch_size=8, steps_per_epoch=1, rotate=True, spacings=[3.0, 10.0])
batch, _ = sampler[0]
d = batch['density'].numpy()
print('batch shape:', d.shape)
assert d.shape[-1] == 2 and np.allclose(d[..., 1], -d[..., 0])
print('ok: channels stay paired')

config2 = {
    'name': 'smoke_smooth', 'output': '/content/runs',
    'data': {'inputs': ['tng300-3/subhalos/L10'], 'labels': 'tng300-3/mask',
             'smoothing': {'sigma_spacing': 0.5}},
    'model': {'depth': 3, 'base_filters': 8},
    'training': {'crop': 64, 'batch_size': 4, 'steps_per_epoch': 20, 'val_batches': 2,
                 'epochs': 1, 'loss': {'name': 'scce', 'class_weights': 'auto'}},
}
out2 = run(config2)
print(open(out2 / 'normalization.json').read())

## 6. Void-vs-rest loss and deep supervision

First the losses on fixed numbers: with `void_weight = 0` the loss equals plain cross-entropy, the void term equals the binary cross-entropy of the void score, and a deep-supervision loss with factor 1 equals cross-entropy. Then a model of each depth has one extra output per coarser level, and a one-epoch run with both options trains, saves a single-output `best.keras`, and scores.

In [ ]:
from pathlib import Path
import numpy as np
from deepvoid.train import run
from deepvoid.losses import DeepSupervisionLoss, WeightedSCCE
from deepvoid.models import main_output, unet3d

rng = np.random.default_rng(0)
y = rng.integers(0, 4, (2, 8, 8, 8))
p = rng.dirichlet(np.ones(4), (2, 8, 8, 8)).astype('float32')
ce = -np.log(np.take_along_axis(p, y[..., None], -1))[..., 0]
bce = -np.where(y == 0, np.log(p[..., 0]), np.log(1 - p[..., 0]))
plain = float(WeightedSCCE()(y, p))
with_void = float(WeightedSCCE(void_weight=0.5)(y, p))
ds1 = float(DeepSupervisionLoss(1)(y, p))
print('plain', plain, 'numpy', ce.mean())
print('with void term', with_void, 'numpy', (ce + 0.5 * bce).mean())
print('deep supervision, factor 1', ds1)
assert abs(plain - ce.mean()) < 1e-4 and abs(with_void - (ce + 0.5 * bce).mean()) < 1e-4
assert abs(ds1 - plain) < 1e-4

for depth in (3, 4, 5):
    m = unet3d(depth=depth, base_filters=4, deep_supervision=True)
    shapes = [tuple(o.shape[1:4]) for o in m(np.zeros((1, 32, 32, 32, 1), 'float32'))]
    print('depth', depth, 'output grids', shapes)
    assert len(shapes) == depth and shapes[-1] == (32 // 2 ** (depth - 1),) * 3
    assert len(main_output(m).outputs) == 1

config3 = {
    'name': 'smoke_deepsup', 'output': '/content/runs',
    'data': {'inputs': ['tng300-3/subhalos/L10'], 'labels': 'tng300-3/mask',
             'smoothing': {'sigma_spacing': 0.5}},
    'model': {'depth': 4, 'base_filters': 8, 'deep_supervision': True},
    'training': {'crop': 64, 'batch_size': 4, 'steps_per_epoch': 20, 'val_batches': 2,
                 'epochs': 1, 'loss': {'name': 'scce', 'class_weights': 'auto', 'void_weight': 1.0}},
}
Path('/content/runs').mkdir(exist_ok=True)
out3 = run(config3)
from deepvoid.models import load_model
print('outputs in best.keras:', len(load_model(out3 / 'best.keras').outputs))
print(open(out3 / 'log.csv').read().splitlines()[0])